# 02 · Data Manipulation with pandas

A single runnable notebook covering pandas essentials for data engineering.
Run cells top to bottom. Dataset: `datasets/sales.csv` (+ `datasets/products.json`).

**Sections:** 1. Introduction · 2. Series · 3. DataFrame · 4. Read/Write ·
5. Accessing & Selecting · 6. Statistics · 7. Manipulation · 8. Cleaning · 9. Merge/Join/Concat

In [1]:
import os
import pandas as pd
import numpy as np

# resolve datasets/ no matter where Jupyter is launched from
DATASETS = next(
    os.path.join(p, "datasets")
    for p in [".", "..", "../..", "../../.."]
    if os.path.isdir(os.path.join(p, "datasets"))
)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)
print("pandas", pd.__version__, "| datasets at", DATASETS)

pandas 3.0.3 | datasets at ../../datasets


## 1. Introduction to pandas

Two core objects: **Series** (1-D labeled array) and **DataFrame** (2-D table).

A **Series** is a single labeled column.

In [2]:
prices = pd.Series(
    [35000, 650, 8500],
    index=["Laptop Pro", "Wireless Mouse", "Standing Desk"],
    name="unit_price",
)
print(prices)
print(f"type={type(prices).__name__}, dtype={prices.dtype}, shape={prices.shape}")

Laptop Pro        35000
Wireless Mouse      650
Standing Desk      8500
Name: unit_price, dtype: int64
type=Series, dtype=int64, shape=(3,)


A **DataFrame** is a table of columns.

In [3]:
df = pd.DataFrame({
    "product":    ["Laptop Pro", "Wireless Mouse", "Standing Desk"],
    "category":   ["Electronics", "Electronics", "Furniture"],
    "unit_price": [35000, 650, 8500],
    "stock":      [50, 200, 25],
})
df

,product,category,unit_price,stock
0,Laptop Pro,Electronics,35000,50
1,Wireless Mouse,Electronics,650,200
2,Standing Desk,Furniture,8500,25


A DataFrame column **is** a Series.

In [4]:
print(type(df["unit_price"]).__name__)

Series


pandas is **vectorized** — operate on a whole column, no loops.

In [5]:
df["stock_value"] = df["unit_price"] * df["stock"]
df[["product", "unit_price", "stock", "stock_value"]]

,product,unit_price,stock,stock_value
0,Laptop Pro,35000,50,1750000
1,Wireless Mouse,650,200,130000
2,Standing Desk,8500,25,212500


## 2. Pandas Series

Create a Series.

In [6]:
s = pd.Series([10, 20, 30, 40])
print(s)
print(f"values={s.values}, dtype={s.dtype}")

0    10
1    20
2    30
3    40
dtype: int64
values=[10 20 30 40], dtype=int64


Custom index — labels instead of positions.

In [7]:
prices = pd.Series([35000, 650, 8500], index=["Laptop", "Mouse", "Desk"])
prices

Laptop    35000
Mouse       650
Desk       8500
dtype: int64

In [8]:
prices["Laptop"]

np.int64(35000)

Label (`.loc`) vs position (`.iloc`).

In [9]:
print(f".loc['Mouse']: {prices.loc['Mouse']}")
print(f".iloc[0]:      {prices.iloc[0]}")

.loc['Mouse']: 650
.iloc[0]:      35000


Vectorized operations.

In [10]:
print((prices * 1.07).round(0))   # add 7% VAT
print(prices > 1000)

Laptop    37450.0
Mouse       696.0
Desk       9095.0
dtype: float64
Laptop     True
Mouse     False
Desk       True
dtype: bool


Index **alignment** — operations match by label, not position (missing labels → `NaN`).

In [11]:
a = pd.Series([1, 2], index=["x", "y"])
b = pd.Series([10, 20], index=["y", "z"])
a + b   # x=NaN, y=12, z=NaN — aligned by index, not position

x     NaN
y    12.0
z     NaN
dtype: float64

## 3. Pandas DataFrame

Create from a dict of columns.

In [12]:
df = pd.DataFrame({
    "product":    ["Laptop Pro", "Wireless Mouse", "Standing Desk", "Ergonomic Chair"],
    "category":   ["Electronics", "Electronics", "Furniture", "Furniture"],
    "unit_price": [35000, 650, 8500, 15000],
    "stock":      [50, 200, 25, 40],
})
df

,product,category,unit_price,stock
0,Laptop Pro,Electronics,35000,50
1,Wireless Mouse,Electronics,650,200
2,Standing Desk,Furniture,8500,25
3,Ergonomic Chair,Furniture,15000,40


Basic info — shape, columns, dtypes.

In [13]:
print(f"shape={df.shape}, columns={list(df.columns)}")
df.dtypes

shape=(4, 4), columns=['product', 'category', 'unit_price', 'stock']


product         str
category        str
unit_price    int64
stock         int64
dtype: object

Single `[]` returns a **Series**; double `[[]]` returns a **DataFrame**.

In [14]:
print(type(df["unit_price"]).__name__)     # Series
print(type(df[["unit_price"]]).__name__)   # DataFrame

Series
DataFrame


Add / modify columns.

In [15]:
df["stock_value"] = df["unit_price"] * df["stock"]
df["needs_reorder"] = df["stock"] < 30
df["price_tier"] = df["unit_price"].apply(lambda p: "premium" if p >= 10000 else "standard")
df

,product,category,unit_price,stock,stock_value,needs_reorder,price_tier
0,Laptop Pro,Electronics,35000,50,1750000,False,premium
1,Wireless Mouse,Electronics,650,200,130000,False,standard
2,Standing Desk,Furniture,8500,25,212500,True,standard
3,Ergonomic Chair,Furniture,15000,40,600000,False,premium


Index operations — `set_index` / `loc`.

In [16]:
df.set_index("product").loc["Laptop Pro"]

category         Electronics
unit_price             35000
stock                     50
stock_value          1750000
needs_reorder          False
price_tier           premium
Name: Laptop Pro, dtype: object

## 4. Reading & Writing Data

Read CSV (parse the date column).

In [17]:
df = pd.read_csv(os.path.join(DATASETS, "sales.csv"), parse_dates=["date"])
print(f"shape={df.shape}")
df.head()

shape=(54, 7)


,date,product,category,quantity,unit_price,revenue,region
0,2024-01-02,Laptop Pro,Electronics,3,35000,105000,Bangkok
1,2024-01-02,Wireless Mouse,Electronics,15,650,9750,Bangkok
2,2024-01-03,Standing Desk,Furniture,2,8500,17000,Chiang Mai
3,2024-01-03,Ergonomic Chair,Furniture,4,15000,60000,Bangkok
4,2024-01-04,Running Shoes,Clothing,10,2800,28000,Phuket


Read JSON and flatten the nested product list.

In [18]:
products = pd.json_normalize(pd.read_json(os.path.join(DATASETS, "products.json"))["products"])
products[["id", "name", "category", "unit_price"]].head()

,id,name,category,unit_price
0,P001,Laptop Pro,Electronics,35000
1,P002,Wireless Mouse,Electronics,650
2,P003,Mechanical Keyboard,Electronics,2500
3,P004,Monitor 27in,Electronics,12000
4,P005,Webcam,Electronics,3500


Useful `read_csv` options — `usecols`, `nrows`.

In [19]:
pd.read_csv(os.path.join(DATASETS, "sales.csv"), usecols=["date", "product", "revenue"], nrows=5)

,date,product,revenue
0,2024-01-02,Laptop Pro,105000
1,2024-01-02,Wireless Mouse,9750
2,2024-01-03,Standing Desk,17000
3,2024-01-03,Ergonomic Chair,60000
4,2024-01-04,Running Shoes,28000


Write CSV / JSON / Parquet.

In [20]:
df.to_csv("/tmp/sales_out.csv", index=False)
df.to_json("/tmp/sales_out.json", orient="records", date_format="iso")
df.to_parquet("/tmp/sales_out.parquet", index=False)   # columnar + compressed, preferred for DE
print("wrote csv / json / parquet to /tmp")

wrote csv / json / parquet to /tmp


Parquet round-trip preserves dtypes (note `datetime64`, not `object`).

In [21]:
pd.read_parquet("/tmp/sales_out.parquet").dtypes

date          datetime64[us]
product                  str
category                 str
quantity               int64
unit_price             int64
revenue                int64
region                   str
dtype: object

## 5. Accessing & Selecting Data

In [22]:
df = pd.read_csv(os.path.join(DATASETS, "sales.csv"), parse_dates=["date"])

Select columns.

In [23]:
df[["product", "revenue"]].head()

,product,revenue
0,Laptop Pro,105000
1,Wireless Mouse,9750
2,Standing Desk,17000
3,Ergonomic Chair,60000
4,Running Shoes,28000


Boolean filtering (combine with `&`).

In [24]:
print(f"Electronics rows: {len(df[df['category'] == 'Electronics'])}")
df[(df["region"] == "Bangkok") & (df["category"] == "Electronics")].head()

Electronics rows: 24


,date,product,category,quantity,unit_price,revenue,region
0,2024-01-02,Laptop Pro,Electronics,3,35000,105000,Bangkok
1,2024-01-02,Wireless Mouse,Electronics,15,650,9750,Bangkok
5,2024-01-05,Monitor 27in,Electronics,5,12000,60000,Bangkok
11,2024-01-14,Webcam,Electronics,12,3500,42000,Bangkok
16,2024-01-21,Mechanical Keyboard,Electronics,5,2500,12500,Bangkok


`isin()` — match any of several values.

In [25]:
len(df[df["region"].isin(["Bangkok", "Chiang Mai"])])

34

`query()` — readable filters.

In [26]:
df.query("revenue > 50000 and category == 'Electronics'")[["date", "product", "revenue"]]

,date,product,revenue
0,2024-01-02,Laptop Pro,105000
5,2024-01-05,Monitor 27in,60000
8,2024-01-10,Laptop Pro,70000
19,2024-01-25,Laptop Pro,140000
23,2024-02-03,Monitor 27in,84000
27,2024-02-10,Laptop Pro,70000
37,2024-02-28,Laptop Pro,105000
38,2024-03-01,Webcam,52500
44,2024-03-13,Laptop Pro,175000
48,2024-03-21,Monitor 27in,72000


`loc` (labels) vs `iloc` (positions).

In [27]:
df.loc[0:2, ["product", "quantity", "revenue"]]

,product,quantity,revenue
0,Laptop Pro,3,105000
1,Wireless Mouse,15,9750
2,Standing Desk,2,17000


In [28]:
df.iloc[0:3, [1, 3, 5]]

,product,quantity,revenue
0,Laptop Pro,3,105000
1,Wireless Mouse,15,9750
2,Standing Desk,2,17000


## 6. Basic Statistics

In [29]:
df = pd.read_csv(os.path.join(DATASETS, "sales.csv"), parse_dates=["date"])
rev = df["revenue"]

Descriptive statistics.

In [30]:
df[["quantity", "unit_price", "revenue"]].describe().round(2)

,quantity,unit_price,revenue
count,54.00,54.00,54.00
mean,16.37,8665.93,41691.67
std,16.98,11323.48,37194.17
min,1.00,120.00,6000.00
25%,4.00,710.00,14187.50
50%,9.00,3200.00,29000.00
75%,25.00,12000.00,55125.00
max,80.00,35000.00,175000.00


Central tendency & spread.

In [31]:
print(f"mean={rev.mean():,.0f}  median={rev.median():,.0f}  std={rev.std():,.0f}")
print(f"IQR={rev.quantile(0.75) - rev.quantile(0.25):,.0f}")

mean=41,692  median=29,000  std=37,194
IQR=40,938


`value_counts` — frequency of each value.

In [32]:
df["region"].value_counts()

region
Bangkok       24
Chiang Mai    10
Phuket         9
Khon Kaen      7
Ayutthaya      4
Name: count, dtype: int64

Correlation between variables.

In [33]:
df[["quantity", "unit_price", "revenue"]].corr().round(3)

,quantity,unit_price,revenue
quantity,1.000,-0.531,-0.448
unit_price,-0.531,1.000,0.856
revenue,-0.448,0.856,1.000


## 7. Data Manipulation

In [34]:
df = pd.read_csv(os.path.join(DATASETS, "sales.csv"), parse_dates=["date"])
df["month"] = df["date"].dt.strftime("%Y-%m")

`map` — element-wise replacement via dict.

In [35]:
df["region"].map({"Bangkok": "BKK", "Phuket": "HKT", "Chiang Mai": "CNX"}).value_counts()

region
BKK    24
CNX    10
HKT     9
Name: count, dtype: int64

`apply` — run a function on each element.

In [36]:
df["revenue"].apply(lambda r: "high" if r >= 50000 else "normal").value_counts()

revenue
normal    38
high      16
Name: count, dtype: int64

`groupby` — aggregate one column several ways.

In [44]:
df.groupby("category")["revenue"].agg(total="sum", mean="mean", count="count").sort_values("total", ascending=False).round(0)

,total,mean,count
category,,,
Electronics,1408500,58688.0,24
Furniture,442600,36883.0,12
Clothing,292100,36512.0,8
Accessories,108150,10815.0,10


`groupby` — multiple named aggregations.

In [47]:
df.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    total_qty=("quantity", "sum"),
    transactions=("revenue", "count"),
).sort_values("total_revenue", ascending=False)

,total_revenue,total_qty,transactions
region,,,
Bangkok,1304600,306,24
Phuket,372000,125,9
Chiang Mai,358800,155,10
Khon Kaen,166950,132,7
Ayutthaya,49000,166,4


Pivot table.

In [48]:
df.pivot_table(values="revenue", index="region", columns="category", aggfunc="sum", fill_value=0).astype(int)

category,Accessories,Clothing,Electronics,Furniture
region,,,,
Ayutthaya,16800,0,13000,19200
Bangkok,30000,99350,879750,295500
Chiang Mai,6000,73150,224250,55400
Khon Kaen,23850,35600,99000,8500
Phuket,31500,84000,192500,64000


Sorting by two keys.

In [40]:
df.sort_values(["region", "revenue"], ascending=[True, False]).head(3)[["region", "product", "revenue"]]

,region,product,revenue
10,Ayutthaya,Bookshelf,19200
20,Ayutthaya,Wireless Mouse,13000
49,Ayutthaya,Notebook,9600


## 8. Data Cleaning

We inject some 'dirt' into a copy, then fix it step by step.

In [41]:
df = pd.read_csv(os.path.join(DATASETS, "sales.csv"))
dirty = df.head(8).copy()
dirty.loc[1, "region"] = "  bangkok  "       # whitespace + casing
dirty.loc[2, "quantity"] = np.nan            # missing value
dirty.loc[3, "category"] = "electronics"     # inconsistent casing
dirty = pd.concat([dirty, dirty.iloc[[0]]], ignore_index=True)   # duplicate row
dirty[["product", "category", "quantity", "region"]]

,product,category,quantity,region
0,Laptop Pro,Electronics,3.0,Bangkok
1,Wireless Mouse,Electronics,15.0,bangkok
2,Standing Desk,Furniture,NaN,Chiang Mai
3,Ergonomic Chair,electronics,4.0,Bangkok
4,Running Shoes,Clothing,10.0,Phuket
5,Monitor 27in,Electronics,5.0,Bangkok
6,Mechanical Keyboard,Electronics,8.0,Chiang Mai
7,Water Bottle,Accessories,25.0,Khon Kaen
8,Laptop Pro,Electronics,3.0,Bangkok


**1. Missing values** — count, then fill.

In [42]:
print(dirty.isnull().sum()[lambda s: s > 0])
dirty["quantity"] = dirty["quantity"].fillna(dirty["quantity"].median())

quantity    1
dtype: int64


**2. Duplicate rows** — detect & drop.

In [43]:
print(f"duplicates: {dirty.duplicated().sum()}")
dirty = dirty.drop_duplicates().reset_index(drop=True)
print(f"rows now: {len(dirty)}")

duplicates: 1
rows now: 8


**3. Fix data types.**

In [44]:
dirty["quantity"] = dirty["quantity"].astype(int)
dirty["quantity"].dtype

dtype('int64')

**4. Normalize strings** — strip + title case.

In [45]:
dirty["region"] = dirty["region"].str.strip().str.title()
dirty["category"] = dirty["category"].str.strip().str.title()
dirty[["category", "region"]].drop_duplicates()

,category,region
0,Electronics,Bangkok
2,Furniture,Chiang Mai
4,Clothing,Phuket
6,Electronics,Chiang Mai
7,Accessories,Khon Kaen


**5. Detect outliers** (IQR rule).

In [46]:
q1, q3 = df["revenue"].quantile([0.25, 0.75])
upper = q3 + 1.5 * (q3 - q1)
print(f"upper fence: {upper:,.0f}  ->  {len(df[df['revenue'] > upper])} outliers")

upper fence: 116,531  ->  3 outliers


## 9. Merge, Join & Concatenate

Load a fact table (sales) and a dimension table (products).

In [47]:
sales = pd.read_csv(os.path.join(DATASETS, "sales.csv"), parse_dates=["date"])
products = pd.json_normalize(pd.read_json(os.path.join(DATASETS, "products.json"))["products"])
print(f"sales={sales.shape}, products={products.shape}")

sales=(54, 7), products=(13, 6)


Inner merge — enrich sales with supplier & stock.

In [48]:
enriched = sales.merge(
    products[["name", "supplier", "stock"]],
    left_on="product", right_on="name", how="inner",
).drop(columns="name")
enriched[["product", "supplier", "stock", "revenue"]].head()

,product,supplier,stock,revenue
0,Laptop Pro,TechCorp,50,105000
1,Wireless Mouse,PeripheralCo,200,9750
2,Standing Desk,FurniturePro,25,17000
3,Ergonomic Chair,FurniturePro,40,60000
4,Running Shoes,SportWear,100,28000


Join types on small demo tables.

In [49]:
left = pd.DataFrame({"id": [1, 2, 3], "name": ["Alice", "Bob", "Carol"]})
right = pd.DataFrame({"id": [2, 3, 4], "city": ["Bangkok", "Phuket", "Chiang Mai"]})
left.merge(right, on="id", how="outer", indicator=True)

,id,name,city,_merge
0,1,Alice,NaN,left_only
1,2,Bob,Bangkok,both
2,3,Carol,Phuket,both
3,4,NaN,Chiang Mai,right_only


`concat` — stack rows vertically.

In [50]:
jan = sales[sales["date"].dt.month == 1]
feb = sales[sales["date"].dt.month == 2]
print(f"jan={len(jan)} + feb={len(feb)} -> {len(pd.concat([jan, feb], ignore_index=True))}")

jan=22 + feb=16 -> 38


Practical — revenue by supplier (uses the merge above).

In [51]:
enriched.groupby("supplier")["revenue"].sum().sort_values(ascending=False).astype(int)

supplier
TechCorp        962500
FurniturePro    442600
DisplayTech     300000
SportWear       292100
PeripheralCo    146000
OfficePro        56400
HealthGear       51750
Name: revenue, dtype: int64